In [1]:
#! pip install from statsmodels
from statsmodels.stats.api import DescrStatsW


In [ ]:
from nemo.collections.tts.models import T5TTS_Model, T5TTS_Discriminator
from nemo.collections.tts.data.text_to_speech_dataset import T5TTSDataset, DatasetSample
from omegaconf.omegaconf import OmegaConf, open_dict
import torch
import os
import soundfile as sf
from IPython.display import display, Audio
import numpy as np
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

### Checkpoint Paths

In [17]:
fs_12_5 = True
dpo_21 = False
dpo_compare = True
dpo_on = False
if fs_12_5:
    hparams_file = "/datap/misc/continuouscheckpoints/lt/localtransformer//koel_12.5_FPS_causal_13codebooks_codecmodel_context5sec_LTN1_hparams.yaml"
    checkpoint_file = "/datap/misc/continuouscheckpoints/lt/localtransformer/koel_12.5_FPS_causal_13codebooks_codecmodel_context5sec_LTN1_epoch289.ckpt"
    #checkpoint_file = "/datap/misc/continuouscheckpoints/lt/localtransformer/koel_12.5_FPS_causal_13codebooks_codecmodel_context5sec_LTN3_epoch270.ckpt"
    #hparams_file = "/datap/misc/continuouscheckpoints/lt/localtransformer/koel_12.5_FPS_causal_13codebooks_codecmodel_context5sec_LTN3_hparams_old.yaml"
    codecmodel_path = "/datap/misc/checkpoints/12.5_FPS_causal_13codebooks_codecmodel.nemo"


    disc_checkpoint_file = "disc_training_frozen_audio_emb_heads4_dim128/T5TTS_Discriminator/0/checkpoints/T5TTS_Discriminator--val_loss=0.3054-epoch=177.ckpt"
    disc_hparams_file = "disc_training_frozen_audio_emb_heads4_dim128/T5TTS_Discriminator/0/hparams.yaml"
else: # 21 fps
    codecmodel_path = "/datap/misc/checkpoints/AudioCodec_21Hz_no_eliz.nemo"
    if not dpo_compare:
        checkpoint_file = "/datap/misc/continuouscheckpoints/lt/localtransformer/dc_yum_recipe_withLT_epoch233.ckpt"
        hparams_file = "/datap/misc/continuouscheckpoints/lt/localtransformer/dc_yum_recipe_withLT_hparams_old.yaml"
    else:
        hparams_file = "/datap/misc/continuouscheckpoints/yum_release/hparams.yaml"
        if not dpo_on:
            checkpoint_file = "/datap/misc/continuouscheckpoints/yum_release/pre-dpo--T5TTS--val_loss=5.2892-epoch=29.ckpt"
        else:
            checkpoint_file = "/datap/misc/continuouscheckpoints/yum_release/dpo-T5TTS--val_loss=0.4513-epoch=3.ckpt"
    # 21 Hz discriminator
    disc_checkpoint_file = "disc_21_128/T5TTS_Discriminator/0/checkpoints/T5TTS_Discriminator--val_loss=0.4759-epoch=114.ckpt"
    disc_hparams_file = "disc_21_128/T5TTS_Discriminator/0/hparams.yaml"


# Temp out dir for saving audios
out_dir = "/datap/misc/t5tts_inference_notebook_samples"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

# Load discriminator

In [ ]:
disc_model_cfg = OmegaConf.load(disc_hparams_file).cfg
disc_model_cfg.train_ds = None
disc_model_cfg.validation_ds = None
disc_model = T5TTS_Discriminator(cfg=disc_model_cfg)

disc_weights = torch.load(disc_checkpoint_file, weights_only=False)
disc_model.load_state_dict(disc_weights['state_dict'])

disc_model.cuda()
disc_model.eval()


### Load Model

In [ ]:
model_cfg = OmegaConf.load(hparams_file).cfg

with open_dict(model_cfg):
    model_cfg.codecmodel_path = codecmodel_path
    if hasattr(model_cfg, 'text_tokenizer'):
        # Backward compatibility for models trained with absolute paths in text_tokenizer
        model_cfg.text_tokenizer.g2p.phoneme_dict = "scripts/tts_dataset_files/ipa_cmudict-0.7b_nv23.01.txt"
        model_cfg.text_tokenizer.g2p.heteronyms = "scripts/tts_dataset_files/heteronyms-052722"
        model_cfg.text_tokenizer.g2p.phoneme_probability = 1.0
    model_cfg.train_ds = None
    model_cfg.validation_ds = None


model = T5TTS_Model(cfg=model_cfg)
print("Loading weights from checkpoint")
ckpt = torch.load(checkpoint_file, weights_only=False)
model.load_state_dict(ckpt['state_dict'])
print("Loaded weights.")

model.use_kv_cache_for_inference = True

model.cuda()
model.eval()

### Initialize Dataset class and helper functions

In [ ]:
test_dataset = T5TTSDataset(
    dataset_meta={},
    sample_rate=model_cfg.sample_rate,
    min_duration=0.5,
    max_duration=20,
    codec_model_downsample_factor=model_cfg.codec_model_downsample_factor,
    bos_id=model.bos_id,
    eos_id=model.eos_id,
    context_audio_bos_id=model.context_audio_bos_id,
    context_audio_eos_id=model.context_audio_eos_id,
    audio_bos_id=model.audio_bos_id,
    audio_eos_id=model.audio_eos_id,
    num_audio_codebooks=model_cfg.num_audio_codebooks,
    prior_scaling_factor=None,
    load_cached_codes_if_available=True,
    dataset_type='test',
    tokenizer_config=None,
    load_16khz_audio=model.model_type == 'single_encoder_sv_tts',
    use_text_conditioning_tokenizer=model.use_text_conditioning_encoder,
    pad_context_text_to_max_duration=model.pad_context_text_to_max_duration,
    context_duration_min=model.cfg.get('context_duration_min', 5.0),
    context_duration_max=model.cfg.get('context_duration_max', 5.0),
)
test_dataset.text_tokenizer, test_dataset.text_conditioning_tokenizer = model._setup_tokenizers(model.cfg, mode='test')



def get_audio_duration(file_path):
    with sf.SoundFile(file_path) as audio_file:
        # Calculate the duration
        duration = len(audio_file) / audio_file.samplerate
        return duration

def create_record(text, context_audio_filepath=None, context_text=None):
    dummy_audio_fp = os.path.join(out_dir, "dummy_audio.wav")
    dummy_audio = sf.write(dummy_audio_fp, np.zeros(22050 * 3), 22050)  # 3 seconds of silence
    record = {
        'audio_filepath' : dummy_audio_fp,
        'duration': 3.0,
        'text': text,
        'speaker': "dummy",
    }
    if context_text is not None:
        assert context_audio_filepath is None
        record['context_text'] = context_text
    else:
        assert context_audio_filepath is not None
        record['context_audio_filepath'] = context_audio_filepath
        record['context_audio_duration'] = get_audio_duration(context_audio_filepath)
    
    return record

### Set transcript and context pairs to test

In [21]:
import pprint
# Change sample text and prompt audio/text here
audio_base_dir = "/"
test_entries = [
    create_record(
        text="The recollection of a unique event cannot, so Bergson contends, be wholly constituted by habit, and is in fact something radically different from the memory which is habit.",
        #text="Did you know that hummingbirds are the only birds capable of flying backward? They achieve this unique flight pattern by rotating their wings in a figure-eight pattern",
        #context_text="Speaker and Emotion: | Language:en Dataset:Riva Speaker:Rodney_WIZWIKI |",
        #context_text="Speaker and Emotion: | Language:en Dataset:Riva Speaker:Lindy_WIZWIKI |",

        #context_audio_filepath="/home/rfejgin/contexts/sqam_cd_49_5sec_mono.wav"
        context_audio_filepath="/home/rfejgin/contexts/GTC_FALL_2021_KEYNOTE_V0Only-44khz-16bit-mono_CH07_0042__5s.wav",
    ),
    # create_record(
    #     text="This is a second sentence to test the model. And it should be longer than the first one.",
    #     context_audio_filepath="/home/rfejgin/contexts/GTC_FALL_2021_KEYNOTE_V0Only-44khz-16bit-mono_CH07_0042__5s.wav",
    # ),    
]

data_samples = []
for entry in test_entries:
    dataset_sample = DatasetSample(
        dataset_name="sample",
        manifest_entry=entry,
        audio_dir=audio_base_dir,
        feature_dir=audio_base_dir,
        text=entry['text'],
        speaker=None,
        speaker_index=0,
        tokenizer_names=["english_phoneme"], # Change this for multilingual: "english_phoneme", "spanish_phoneme", "english_chartokenizer", "german_chartokenizer".. 
    )
    data_samples.append(dataset_sample)
    
test_dataset.data_samples = data_samples

test_data_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    collate_fn=test_dataset.collate_fn,
    num_workers=0,
    shuffle=False
)

### Generate With Prior

In [ ]:
import matplotlib.pyplot as plt

def title_with_subtitle(ax, title, subtitle):
    ax.set_title(title, fontsize=16, pad=30)
    ax.text(
        0.5, 1.02, subtitle, 
        transform=ax.transAxes, 
        fontsize=10, 
        ha='center', 
        va='bottom'
    )

item_idx = 0
for bidx, batch in enumerate(test_data_loader):
    print("Processing batch {} out of {}".format(bidx, len(test_data_loader)))
    model.t5_decoder.reset_cache(use_cache=True)
    batch_cuda ={}
    for key in batch:
        if isinstance(batch[key], torch.Tensor):
            batch_cuda[key] = batch[key].cuda()
        else:
            batch_cuda[key] = batch[key]
    import time
    st = time.time()
    accuracies = []
    num_iterations = 10
    for idx in range(num_iterations):
        print(f"\nIteration {idx} / {num_iterations}\n")
        for use_cfg in [False]:
            for resample_threshold in [None]:#, -2.0]:
                for use_local_transformer_for_inference in [True]:#,True]: #, True]:#, True]:
                    for apply_prior in [False]:
                        for resample_and_rank in [False]:#[True, False]:
                            predicted_audio, predicted_audio_lens, predicted_codes, predicted_codes_lens, rtf_metrics, cross_attn_np, all_heads_attn_np, disc_preds, disc_preds_raw,disc_preds_post_sigmoid, resample_counter = model.infer_batch(
                                batch_cuda, 
                                max_decoder_steps=430, 
                                temperature=0.6, 
                                topk=80, 
                                use_cfg=use_cfg,
                                cfg_scale=2.5,
                                prior_epsilon=0.1,
                                lookahead_window_size=5,
                                return_cross_attn_probs=True,
                                estimate_alignment_from_layers=[5],
                                apply_attention_prior=apply_prior,
                                apply_prior_to_layers=[0,1,2,3,4,5,6,7,8,9,10,11],
                                compute_all_heads_attn_maps=True,
                                start_prior_after_n_audio_steps=0,
                                use_local_transformer_for_inference=use_local_transformer_for_inference,
                                discriminator=disc_model,
                                resample_and_rank=resample_and_rank,
                                resample_and_rank_count=10,
                                resample_threshold=resample_threshold,
                            )
                            if False:
                                #print("Discriminator Predictions:", disc_preds)
                                acc = (disc_preds == False).float().sum() / disc_preds.numel()
                                print(f"Discriminator accuracy: {acc*100:.2f}%")
                            print("generation time", time.time() - st)
                            show_attn_maps = False
                            pprint.pprint(rtf_metrics)
                            for idx in range(predicted_audio.size(0)):
                                predicted_audio_np = predicted_audio[idx].float().detach().cpu().numpy()
                                predicted_audio_np = predicted_audio_np[:predicted_audio_lens[idx]]
                                audio_path = os.path.join(out_dir, f"predicted_audio_{item_idx}.wav")
                                sf.write(audio_path, predicted_audio_np, model.cfg.sample_rate)
                                print(test_entries[bidx]['text'])
                                print("Prior Used?", apply_prior)
                                print("use_local_transformer", use_local_transformer_for_inference)
                                print("resample_and_rank?", resample_and_rank)
                                print(f"Mean raw preds: {disc_preds_raw.mean().cpu().item():.2f}")
                                print(f"Min raw preds:  {disc_preds_raw.min().cpu().item():.2f}")
                                print(f"resample_threshold: {resample_threshold}")
                                print(f"resample_counter: {resample_counter}")
                                print(f"reample percentage: {(resample_counter / len(disc_preds_raw))*100:.2f}%")
                                print(f"use_cfg: {use_cfg}")
                                accuracy = ((disc_preds==False).sum() / disc_preds.numel()).item()*100
                                accuracies.append(accuracy)
                                print(f"Accuracy: {accuracy:.2f}%")
                                print(f"checkpoint: {checkpoint_file}")
                                display(Audio(audio_path))
                                item_idx += 1
                                
                                if True:
                                    #disc_preds_raw = disc_preds_raw.cpu().numpy()                    
                                    disc_preds_post_sigmoid = disc_preds_post_sigmoid.cpu().numpy()                    

                                    # Discriminator Predictions
                                    fig, ax = plt.subplots()
                                    ax.plot(disc_preds_post_sigmoid, 'x')
                                    title_with_subtitle(ax, f"predictions", f"use_local_transformer={use_local_transformer_for_inference},\nckpt={os.path.basename(checkpoint_file)}")
                                    plt.show()

                                    # Discriminator Predictions Histogram
                                    fig, ax = plt.subplots()
                                    plt.hist(disc_preds_post_sigmoid)
                                    title_with_subtitle(ax, f"prediction histogram", f"use_local_transformer={use_local_transformer_for_inference},\nckpt={os.path.basename(checkpoint_file)}")
                                    plt.show()
                                if show_attn_maps:
                                    plt.imshow(cross_attn_np[idx])
                                    plt.show()
        #                     for hidx, head_cross_attn in enumerate(all_heads_attn_np[idx]):
        #                         layer_num = hidx // model.cfg.t5_decoder.xa_n_heads
        #                         head_num = hidx % model.cfg.t5_decoder.xa_n_heads
        #                         print("item, layer, head", idx, layer_num, head_num)
        #                         plt.imshow(all_heads_attn_np[idx][hidx])
        #                         plt.show()

                    print("------------------------------------")


In [ ]:
#%pip install from statsmodels
import numpy as np
from statsmodels.stats.api import DescrStatsW

accuracies = np.array(accuracies)

print(f"Average accuracy: {np.mean(accuracies):.2f}%")
print(f"STD of accuracy: {np.std(accuracies):.2f}%")

# Create a DescrStatsW object
descriptive_stats = DescrStatsW(accuracies)

# Get the confidence interval for the mean (default is 95%)
confidence_interval = descriptive_stats.tconfint_mean()
print(f"Confidence interval for the mean: {confidence_interval[0]:.2f}% -- {confidence_interval[1]:.2f}%")

